# Inspect the true process and simulated observations

Follow the two data-generation modules from a regular process through irregular sampling
to uncertain observations; run all cells from the project root.
The original process is generated directly every 20 years, without an extra downsampling stage;
all figures are saved in `figures/01_inspect_simulated_records/`.
Ages use kyr BP relative to 1950 CE, with older ages on the left and time moving toward younger ages.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

if not (Path.cwd() / "toolbox").is_dir():
    raise RuntimeError("Start this notebook from the project root containing toolbox/.")

from toolbox.process import generate_process
from toolbox.observation import (
    make_age_uncertainty,
    sample_irregular_indices,
    simulate_observations,
)

project_root = Path.cwd()
figure_dir = project_root / "figures" / "01_inspect_simulated_records"
figure_dir.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})


## Settings

The current reference model uses a one-step (20-year) coupling delay; intrinsic memory remains separate.
The mapping $a=\exp(-\Delta t/T)$ follows the scalar OU transition
([Särkkä and Solin, 2019](https://users.aalto.fi/~asolin/sde-book/sde-book.pdf));
the delayed coupling below remains a discrete model, not an exact joint SDE solution.


In [ ]:
# True process: values are in arbitrary units (a.u.); 1 kyr = 1000 years.
# Nonnegative integer seed for the process, sample positions, and observation errors.
random_seed = 20261007
# Retained record duration, > 0 kyr; duration / dt must be an integer.
duration = 30.0
# Process step, > 0 kyr; generate directly every 20 years (1501 points over 30 kyr).
# The coefficients and innovation SD below are recalculated when dt changes.
dt = 0.020
# Oldest retained age, kyr BP relative to 1950 CE; 30 kyr BP to 0 for the current duration.
age_start = 30.0
# X-to-Y delay, > 0 kyr with integer lag / dt; one 20-year step in this first-order model.
# Both the self-memory terms and the coupling use values from the previous process step.
lag = 0.020
# X intrinsic exponential memory time, > 0 kyr; larger values give slower autocorrelation decay.
memory_x = 0.5
# Y's intrinsic memory, > 0 kyr; coupling may give the full Y series additional driver memory.
memory_y = 0.2
# Target stationary X SD (a.u.); 0.95 calibrates the previous example's amplitude, not the current AR.
x_stationary_std = 1 / np.sqrt(1 - 0.95**2)
# Target SD of uncoupled Y (a.u.); 0.8 likewise preserves the previous background amplitude.
y_background_std = 1 / np.sqrt(1 - 0.8**2)
# Long-run gain from a constant standardized driver to Y (a.u.).
# Use 4 for the one-step reference benchmark; the background process noise and memory stay fixed.
# This is not a fixed signal-to-noise ratio or a causal score returned by the analysis methods.
coupling_gain = 4.0
# Warm-up length in multiples of the longer memory, > 0; discarded before retaining the record.
burn_in_timescales = 5.0

# Convert physical timescales to the existing discrete-model inputs; keep these calculations visible.
duration_steps = int(round(duration / dt))
if not np.isclose(duration / dt, duration_steps, rtol=0, atol=1e-10):
    raise ValueError("duration / dt must be an integer.")
n_steps = duration_steps + 1  # Include both endpoints of the retained duration.
ar_x = np.exp(-dt / memory_x)  # Memory coefficient per dt, not a correlation probability.
ar_y = np.exp(-dt / memory_y)  # Y's own memory coefficient per dt.
noise_std = (  # Independent Gaussian process innovations, not measurement errors.
    x_stationary_std * np.sqrt(1 - ar_x**2),
    y_background_std * np.sqrt(1 - ar_y**2),
)
coupling = (1 - ar_y) * coupling_gain  # Per-step coefficient of standardized g(X).
burn_in = max(  # Integer warm-up steps; always supply at least the required lag history.
    int(np.ceil(burn_in_timescales * max(memory_x, memory_y) / dt)),
    int(np.ceil(lag / dt)),
)

# Gentle irregular sampling keeps most native points to limit interpolation effects.
# Young-end target gap before grid rounding, > 0 kyr; this is not the observed mean spacing.
young_spacing = 0.0275
# Old-end target gap, >= young_spacing in kyr; equal here to remove age-dependent thinning.
old_spacing = 0.0275
# Uniform gap multiplier in [1-jitter, 1+jitter], with 0 <= jitter < 1.
# At these settings, rounding gives 20/40-year gaps and retains about 95.7% of native points.
# Require young_spacing * (1 - spacing_jitter) >= dt; actual mean spacing is about 20.9 years.
spacing_jitter = 0.10
# Earlier targets of 0.040/0.060 kyr with jitter=0.25 are retained as a future sensitivity case.

# Age scales describe 1 SD, not a maximum possible error; these are large-error illustrations.
# Their amplitudes have not been recalibrated for the one-step model; age-error causal analysis is paused.
# C's constant SD and L/E's old-end SD, >= 0 kyr; 50 years, or 2.5 times the current delay.
age_scale = 0.05
# Dimensionless E curvature, > 0; larger values concentrate growth near the old end at fixed peak SD.
exp_shape = 2.0
# Peak SD of each smooth J interval, >= 0 kyr; 100 years, or 5 times the current delay.
jump_scale = 0.10
# J centers at fixed fractions of duration, in kyr BP; 6, 15, and 24 kyr BP here.
# Keeping fractional centers and coverage means the intervals widen when the record is lengthened.
jump_centers = age_start - duration + duration * np.array([0.2, 0.5, 0.8])
# Nonzero J intervals as fractions of record duration, 0 < fraction <= 1; changes width, not peak count.
# This is time coverage, not a fraction of samples or a probability interval.
jump_coverages = [0.10, 0.30, 0.50]
# Main observation's J coverage; choose from jump_coverages, with higher values affecting more ages.
observation_jump_coverage = 0.30
# Error correlation length, > 0 in normalized sample order [0, 1], not kyr; larger is smoother.
# The previous 10 kyr record used 0.05; inverse-duration scaling approximately preserves local
# physical smoothness under the same sampling gradient, without defining an exact kyr correlation length.
age_correlation_length = 0.05 * 10.0 / duration
# Measurement SD / full true-series SD, >= 0; explore 0-0.5 later, with zero in the main example.
measurement_noise_ratio = 0.0
# Measurement SD ratio used only in the final pairing check; does not change the main observations.
measurement_check_ratio = 0.2
# Detail-panel limits in kyr BP, ordered (old, young); all sampled data remain available.
zoom_age_limits = (15.6, 14.4)

# Derived age bounds: envelope inputs and plot limits use opposite endpoint orders.
# Youngest age in kyr BP; negative ages would fall after 1950 CE.
age_end = age_start - (n_steps - 1) * dt
age_range = (age_end, age_start)  # Full envelope domain in kyr BP: (young, old).
age_limits = (age_start, age_end)  # Plot domain in kyr BP: (old, young), older ages on the left.

# Paired scenarios recreate Generators from these seeds to reuse the same random draws.
(
    process_seed,        # Process innovations, shared by the three systems for paired comparisons.
    sampling_x_seed,     # X sample positions.
    sampling_y_seed,     # Y sample positions, independent of X.
    age_x_seed,          # X age errors, reused across envelope types.
    age_y_seed,          # Y age errors, independent of X.
    measurement_x_seed,  # X measurement errors, reused across age scenarios.
    measurement_y_seed,  # Y measurement errors, independent of X.
) = np.random.SeedSequence(random_seed).spawn(7)


## 1. Generate linear, nonlinear, and independent processes

All systems use the first-order equations $X_t=aX_{t-1}+u_t$ and
$Y_t=bY_{t-1}+c\,g(X_{t-1})+v_t$, with a 20-year step, linear or saturating coupling,
and `coupling=0` for independence.
Compare theoretical amplitudes and memory times with the realized series; subtract the paired
independent Y to inspect the driven component without treating its realized variance ratio as fixed.


In [ ]:
processes = {}
# Both coupling transforms use their theoretical stationary SD, not each sample's realized SD.
for name, form, coefficient in [
    ("Linear", "linear", coupling),
    ("Nonlinear", "nonlinear", coupling),
    ("Independent", "linear", 0.0),
]:
    processes[name] = generate_process(
        n_steps=n_steps, dt=dt, age_start=age_start, lag=lag,
        ar_x=ar_x, ar_y=ar_y, coupling=coefficient,
        coupling_form=form, noise_std=noise_std, burn_in=burn_in,
        rng=np.random.default_rng(process_seed),
    )

process = processes["Linear"]
series = process["series"]
age = series["age"].to_numpy()
x = series["x"].to_numpy()
y = series["y"].to_numpy()

# The paired independent run uses the same Y innovations and supplies the background component.
background_y = processes["Independent"]["series"]["y"].to_numpy()
process_summary = pd.DataFrame([
    {
        "model": name,
        "n_steps": len(result["series"]),
        "x_theoretical_std": x_stationary_std,
        "x_realized_std": result["series"]["x"].std(ddof=0),
        "y_background_theoretical_std": y_background_std,
        "y_background_realized_std": np.std(background_y),
        "y_total_realized_std": result["series"]["y"].std(ddof=0),
    }
    for name, result in processes.items()
])
display(process_summary)

# Exact stationary ratio for this discrete LINEAR driver/filter; delay does not change stationary variance.
linear_theoretical_ratio = (
    coupling_gain**2 * (1 - ar_y) / (1 + ar_y)
    * (1 + ar_x * ar_y) / (1 - ar_x * ar_y) / y_background_std**2
)
coupling_summary = []
for name in ["Linear", "Nonlinear"]:
    driven_y = processes[name]["series"]["y"].to_numpy() - background_y
    coupling_summary.append({
        "model": name,
        "driven_y_realized_std": np.std(driven_y),
        "background_y_realized_std": np.std(background_y),
        "realized_signal_to_background_variance": np.var(driven_y) / np.var(background_y),
        "linear_theoretical_variance_ratio": linear_theoretical_ratio if name == "Linear" else np.nan,
    })
display(pd.DataFrame(coupling_summary))

# Duration / memory is a span diagnostic, not an effective number of independent observations.
timescale_summary = pd.Series({
    "record_duration_kyr": duration,
    "original_step_years": dt * 1000,
    "original_steps_per_delay": lag / dt,
    "record_span_in_x_memory_times": duration / memory_x,
    "warm_up_years": burn_in * dt * 1000,
    "coupling_delay_years": lag * 1000,
    "x_ar_decay_years": memory_x * 1000,
    "y_intrinsic_ar_decay_years": memory_y * 1000,
    "C_L_E_scale_over_delay": age_scale / lag,
    "J_peak_sd_over_delay": jump_scale / lag,
    "target_old_end_intervals_per_delay": lag / old_spacing,
}, name="value").to_frame()
display(timescale_summary)
display(series.head())


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 6), sharex=True)
for ax, (name, result) in zip(axes, processes.items()):
    values = result["series"]
    ax.plot(values["age"], values["x"], lw=0.8, label="X")
    ax.plot(values["age"], values["y"], lw=0.8, alpha=0.8, label="Y")
    ax.set_title(name, loc="left")
    ax.set_ylabel("Value (a.u.)")
    ax.set_xlim(*age_limits)
    ax.legend(loc="upper right", ncol=2)
axes[-1].set_xlabel("Age (kyr BP)")
fig.tight_layout()
fig.savefig(figure_dir / "01_true_processes.png", dpi=180)
fig.savefig(figure_dir / "01_true_processes.pdf")
plt.show()

driver_std = process["truth"]["driver_std"]
nonlinear_std = processes["Nonlinear"]["truth"]["coupling_transform_std"]
driver_values = np.linspace(-3 * driver_std, 3 * driver_std, 301)
linear_response = driver_values / driver_std
nonlinear_response = np.tanh(2 * driver_values / driver_std) / nonlinear_std

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(driver_values, linear_response, label="Linear")
ax.plot(driver_values, nonlinear_response, label="Nonlinear")
ax.set_xlabel("Driver X (a.u.)")
ax.set_ylabel("Coupling transform g(X)")
ax.legend()
fig.tight_layout()
fig.savefig(figure_dir / "02_coupling_transforms.png", dpi=180)
fig.savefig(figure_dir / "02_coupling_transforms.pdf")
plt.show()


## 2. Sample the processes at irregular ages

Keep about 95.7% of the original points, with mostly 20-year gaps and occasional 40-year gaps after grid rounding. X and Y have independent sample locations; the table and plots report the actual spacing and full retained age span.


In [ ]:
sample_x = sample_irregular_indices(
    age, young_spacing=young_spacing, old_spacing=old_spacing,
    jitter=spacing_jitter, rng=np.random.default_rng(sampling_x_seed),
)
sample_y = sample_irregular_indices(
    age, young_spacing=young_spacing, old_spacing=old_spacing,
    jitter=spacing_jitter, rng=np.random.default_rng(sampling_y_seed),
)

sampling_coverage = pd.DataFrame([
    {
        "record": name,
        "n_samples": len(indices),
        "oldest_age_kyr_BP": age[indices[0]],
        "youngest_age_kyr_BP": age[indices[-1]],
        "sampled_span_kyr": age[indices[0]] - age[indices[-1]],
    }
    for name, indices in [("X", sample_x), ("Y", sample_y)]
])
display(sampling_coverage)
# The last grid point is not forced into the sample; the final age may be slightly above zero.
# Samples per delay describe temporal resolution, not independent information.
sampling_summary = pd.DataFrame([
    {
        "record": name,
        "fraction_retained": len(indices) / len(age),
        "min_spacing_kyr": np.min(-np.diff(age[indices])),
        "median_spacing_kyr": np.median(-np.diff(age[indices])),
        "max_gap_kyr": np.max(-np.diff(age[indices])),
        "mean_spacing_kyr": np.mean(-np.diff(age[indices])),
        "fraction_long_gaps": np.mean(-np.diff(age[indices]) > dt * (1 + 1e-8)),
        "samples_per_true_lag": lag / np.median(-np.diff(age[indices])),
    }
    for name, indices in [("X", sample_x), ("Y", sample_y)]
])
display(sampling_summary)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 5.5), sharex="col", sharey="row")
for row, (name, values, indices) in enumerate([("X", x, sample_x), ("Y", y, sample_y)]):
    for col, limits in enumerate([age_limits, zoom_age_limits]):
        ax = axes[row, col]
        ax.plot(age, values, color="0.7", lw=0.8, label="True process")
        ax.scatter(age[indices], values[indices], s=6 if col == 0 else 20,
                   label="Samples", zorder=3)
        ax.set_xlim(*limits)
    axes[row, 0].axvspan(zoom_age_limits[1], zoom_age_limits[0],
                        color="0.8", alpha=0.4, label="Detail window")
    axes[row, 0].set_ylabel(f"{name} (a.u.)")
axes[0, 0].set_title("Full sampled record", loc="left")
axes[0, 1].set_title(
    f"Detail: {zoom_age_limits[0]:g} to {zoom_age_limits[1]:g} kyr BP", loc="left",
)
axes[0, 0].legend(loc="upper right", fontsize=8)
axes[0, 1].legend(loc="upper right", ncol=2, fontsize=8)
for ax in axes[-1]:
    ax.set_xlabel("Age (kyr BP)")
fig.tight_layout()
fig.savefig(figure_dir / "03_irregular_samples.png", dpi=180)
fig.savefig(figure_dir / "03_irregular_samples.pdf")
plt.show()

fig, ax = plt.subplots(figsize=(11, 3))
for name, indices in [("X", sample_x), ("Y", sample_y)]:
    sample_age = age[indices]
    interval_mid_age = (sample_age[:-1] + sample_age[1:]) / 2
    ax.scatter(interval_mid_age, -np.diff(sample_age), s=10, alpha=0.6, label=name)
mean_spacing = young_spacing + (old_spacing - young_spacing) * (age - age_end) / (age_start - age_end)
ax.plot(age, mean_spacing, color="black", lw=1.5, label="Target before grid rounding")
ax.set_xlim(*age_limits)
ax.set_xlabel("Age (kyr BP)")
ax.set_ylabel("Sampling interval (kyr)")
ax.legend(ncol=3)
fig.tight_layout()
fig.savefig(figure_dir / "04_sampling_intervals.png", dpi=180)
fig.savefig(figure_dir / "04_sampling_intervals.pdf")
plt.show()


## 3. Inspect age-uncertainty envelopes

C is constant, L grows linearly, E grows faster near the old end, and J has
smooth bell-shaped peaks within finite intervals.
These curves show one standard deviation, not age jumps or probability densities;
wider J intervals increase both affected duration and total uncertainty.
The 50/100-year scales are large-error illustrations (2.5/5 times the 20-year delay);
these amplitudes are not yet recalibrated for the main experiment, whose age-error causal analysis is paused.


In [ ]:
# J uses exp(-z**2 / (1 - z**2)) for abs(z) < 1, zero elsewhere;
# z is distance from the center divided by interval half-width, not a random variable.
# This idealized age-error envelope is not an age-model posterior from dating control points.
jump_intervals_by_coverage = {}
for coverage in jump_coverages:
    interval_width = coverage * (age_start - age_end) / len(jump_centers)
    jump_intervals_by_coverage[coverage] = [
        (center - interval_width / 2, center + interval_width / 2)
        for center in jump_centers
    ]
jump_intervals = jump_intervals_by_coverage[observation_jump_coverage]

# Equal scale gives equal constant/old-end SD, not equal total uncertainty across C, L, and E.
age_envelopes = {}
for kind in ["C", "L", "E"]:
    age_envelopes[kind] = make_age_uncertainty(
        age, kind=kind, scale=age_scale, age_range=age_range, exp_shape=exp_shape,
    )
age_envelopes["J"] = make_age_uncertainty(
    age, kind="J", scale=jump_scale, age_range=age_range, jump_intervals=jump_intervals,
)

fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
for kind, envelope in age_envelopes.items():
    axes[0].plot(age, envelope, label=kind)
axes[0].set_title(f"Four uncertainty envelopes; J coverage = {observation_jump_coverage:.0%}", loc="left")
for coverage, intervals in jump_intervals_by_coverage.items():
    envelope = make_age_uncertainty(
        age, kind="J", scale=jump_scale, age_range=age_range, jump_intervals=intervals,
    )
    axes[1].plot(age, envelope, label=f"J coverage = {coverage:.0%}")
for ax in axes:
    ax.set_xlim(*age_limits)
    ax.set_ylabel("Age standard deviation (kyr)")
    ax.legend(ncol=4, loc="upper right")
axes[-1].set_xlabel("Age (kyr BP)")
fig.tight_layout()
fig.savefig(figure_dir / "05_age_uncertainty_envelopes.png", dpi=180)
fig.savefig(figure_dir / "05_age_uncertainty_envelopes.pdf")
plt.show()

display(pd.DataFrame([
    {
        "target_coverage": coverage,
        "actual_time_coverage": sum(old - young for young, old in intervals) / (age_start - age_end),
        "n_intervals": len(intervals),
        "peak_std_kyr": jump_scale,
    }
    for coverage, intervals in jump_intervals_by_coverage.items()
]))


## 4. Simulate observations with L for X and J for Y

Keep each sampled value attached to its original sample and perturb its assigned age,
with correlated errors within each record and independent errors between records.
`observed` contains analysis inputs with potentially reversed age order, while
`truth` is reserved for validation; ordered age ensembles will be handled separately.


In [ ]:
age_std_x = make_age_uncertainty(
    age[sample_x], kind="L", scale=age_scale, age_range=age_range,
)
age_std_y = make_age_uncertainty(
    age[sample_y], kind="J", scale=jump_scale, age_range=age_range,
    jump_intervals=jump_intervals,
)

record_x = simulate_observations(
    age, x, sample_x, age_std_x,
    age_correlation_length=age_correlation_length,
    measurement_noise_ratio=measurement_noise_ratio,
    rng_age=np.random.default_rng(age_x_seed),
    rng_measurement=np.random.default_rng(measurement_x_seed),
)
record_y = simulate_observations(
    age, y, sample_y, age_std_y,
    age_correlation_length=age_correlation_length,
    measurement_noise_ratio=measurement_noise_ratio,
    rng_age=np.random.default_rng(age_y_seed),
    rng_measurement=np.random.default_rng(measurement_y_seed),
)

print("X: information available to analysis")
display(record_x["observed"].head())
print("X: simulation truth, kept separate")
display(record_x["truth"].head())

observation_summary = pd.DataFrame([
    {
        "record": name,
        "n_samples": len(record["observed"]),
        "age_order_violations": int(np.sum(np.diff(record["observed"]["age_obs"]) >= 0)),
        "age_error_rms_kyr": np.sqrt(np.mean(record["truth"]["age_error"] ** 2)),
        "measurement_error_rms": np.sqrt(np.mean(record["truth"]["measurement_error"] ** 2)),
    }
    for name, record in [("X: L", record_x), ("Y: J", record_y)]
])
display(observation_summary)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 6))
for row, (name, record) in enumerate([("X: L", record_x), ("Y: J", record_y)]):
    observed = record["observed"]
    truth = record["truth"]

    # Scatter preserves sample identity without implying that the observed ages are ordered.
    ax = axes[row, 0]
    ax.plot(truth["age_true"], truth["value_true"], color="0.6", lw=1, label="True sample ages")
    ax.scatter(observed["age_obs"], observed["value_obs"], s=16, label="Observed ages", zorder=3)
    ax.set_xlim(*zoom_age_limits)
    ax.set_title(f"{name}: detail view", loc="left")
    ax.set_ylabel("Value (a.u.)")
    ax.legend(loc="upper right")

    ax = axes[row, 1]
    ax.fill_between(truth["age_true"], -observed["age_std"], observed["age_std"],
                    color="0.85", label="Input +/- 1 SD")
    ax.plot(truth["age_true"], truth["age_error"], lw=1, label="One error realization")
    ax.axhline(0, color="0.4", lw=0.7)
    ax.set_xlim(*age_limits)
    ax.set_ylabel("Observed - true age (kyr)")
    ax.legend(loc="upper right")
for ax in axes.flat:
    ax.set_xlabel("Age (kyr BP)")
fig.tight_layout()
fig.savefig(figure_dir / "06_simulated_observations.png", dpi=180)
fig.savefig(figure_dir / "06_simulated_observations.pdf")
plt.show()


## 5. Check the data conventions

Check time direction, sample identity, exact recovery with zero error, and reproducibility.
A single error realization need not have exactly zero mean or stay within one standard deviation.


In [ ]:
assert np.all(np.diff(series["time"]) > 0)
assert np.all(np.diff(age) < 0)
np.testing.assert_allclose(age, age_start - series["time"].to_numpy())
for indices in [sample_x, sample_y]:
    assert np.all(np.diff(indices) > 0)
    assert indices[0] >= 0 and indices[-1] < len(age)

record_zero = simulate_observations(
    age, x, sample_x, np.zeros(len(sample_x)),
    age_correlation_length=age_correlation_length, measurement_noise_ratio=0.0,
    rng_age=np.random.default_rng(age_x_seed),
    rng_measurement=np.random.default_rng(measurement_x_seed),
)
np.testing.assert_array_equal(record_zero["observed"]["age_obs"], age[sample_x])
np.testing.assert_array_equal(record_zero["observed"]["value_obs"], x[sample_x])
np.testing.assert_array_equal(record_zero["truth"]["source_index"], sample_x)

process_repeat = generate_process(
    n_steps=n_steps, dt=dt, age_start=age_start, lag=lag,
    ar_x=ar_x, ar_y=ar_y, coupling=coupling, coupling_form="linear",
    noise_std=noise_std, burn_in=burn_in, rng=np.random.default_rng(process_seed),
)
pd.testing.assert_frame_equal(series, process_repeat["series"])
record_repeat = simulate_observations(
    age, x, sample_x, age_std_x,
    age_correlation_length=age_correlation_length,
    measurement_noise_ratio=measurement_noise_ratio,
    rng_age=np.random.default_rng(age_x_seed),
    rng_measurement=np.random.default_rng(measurement_x_seed),
)
pd.testing.assert_frame_equal(record_x["observed"], record_repeat["observed"])
pd.testing.assert_frame_equal(record_x["truth"], record_repeat["truth"])

exact_j = record_y["observed"]["age_std"] == 0
np.testing.assert_array_equal(
    record_y["observed"].loc[exact_j, "age_obs"],
    record_y["truth"].loc[exact_j, "age_true"],
)
print("Passed: time direction, ordered sample indices, zero-error recovery, reproducibility, exact J ages.")


## 6. Check that changing age errors preserves observed values

Generate paired L and J observations with measurement noise at 20% of the full X-series
standard deviation, reusing the same age and measurement seeds.
This checks pairing without replacing the main zero-measurement-error observations.


In [ ]:
paired_records = {}
for kind, scale in [("L", age_scale), ("J", jump_scale)]:
    envelope = make_age_uncertainty(
        age[sample_x], kind=kind, scale=scale, age_range=age_range,
        jump_intervals=jump_intervals if kind == "J" else None,
    )
    paired_records[kind] = simulate_observations(
        age, x, sample_x, envelope,
        age_correlation_length=age_correlation_length,
        measurement_noise_ratio=measurement_check_ratio,
        rng_age=np.random.default_rng(age_x_seed),
        rng_measurement=np.random.default_rng(measurement_x_seed),
    )

np.testing.assert_array_equal(
    paired_records["L"]["observed"]["value_obs"],
    paired_records["J"]["observed"]["value_obs"],
)
np.testing.assert_array_equal(
    paired_records["L"]["truth"]["source_index"],
    paired_records["J"]["truth"]["source_index"],
)
display(pd.DataFrame([
    {
        "age_type": kind,
        "n_samples": len(record["observed"]),
        "measurement_std": record["observed"]["measurement_std"].iloc[0],
        "realized_measurement_rms": np.sqrt(np.mean(record["truth"]["measurement_error"] ** 2)),
        "realized_age_rms_kyr": np.sqrt(np.mean(record["truth"]["age_error"] ** 2)),
    }
    for kind, record in paired_records.items()
]))
print("Passed: changing the age envelope preserves sample positions and observed proxy values.")


## Objects retained for inspection

Inspect `process`, `sample_x/sample_y`, `age_std_x/age_std_y`, and `record_x/record_y`
to trace how sample positions, age uncertainty, and observed values relate.
Later age-ensemble analysis should use `observed` and the required error-model metadata,
keeping `truth` separate for evaluation.
